# Xeno-Canto Bird Classification — Final Notebook (YAMNet → XGBoost)

This notebook is designed for **presentation + repo**:
- Loads cached feature matrix (**emb_mean + emb_std + n_frames** → 2049 features)
- Loads a trained **XGBoost** model (recommended) and evaluates it
- Runs a **Top‑5 demo** from a single `.npz` embedding file
- Produces concise metrics suitable for reporting

> Warning: heavy artifacts (audio, embeddings, big caches) out of Git


## 0) Setup

Adjust the `PROJECT_ROOT` and paths if needed.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

# ==========
# PATHS
# ==========
PROJECT_ROOT = Path(r"C:\Projects\Xeno_Canto_Project")
EMB_DIR = PROJECT_ROOT / "artifacts" / "embeddings_yamnet"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "model_artifacts"


# Cached dataset (created previously)
X_CACHE = ARTIFACTS_DIR / "X_2049.npy"
Y_CACHE = ARTIFACTS_DIR / "y.npy"

# Model artifacts (save/load)
XGB_MODEL_PATH = ARTIFACTS_DIR / "xgb_model.joblib"
LABELS_PATH = ARTIFACTS_DIR / "label_encoder.joblib"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EMB_DIR:", EMB_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("X_CACHE exists:", X_CACHE.exists())
print("Y_CACHE exists:", Y_CACHE.exists())
print("XGB_MODEL_PATH exists:", XGB_MODEL_PATH.exists())
print("LABELS_PATH exists:", LABELS_PATH.exists())


PROJECT_ROOT: C:\Projects\Xeno_Canto_Project
EMB_DIR: C:\Projects\Xeno_Canto_Project\artifacts\embeddings_yamnet
ARTIFACTS_DIR: C:\Projects\Xeno_Canto_Project\artifacts\model_artifacts
X_CACHE exists: True
Y_CACHE exists: True
XGB_MODEL_PATH exists: False
LABELS_PATH exists: True


## 1) Load cached dataset (fast)

This avoids scanning ~90k `.npz` files every run.


In [2]:
# Load cached dataset (memory-mapped for stability on 16GB RAM)
X = np.load(X_CACHE, mmap_mode="r")
y = np.load(Y_CACHE, allow_pickle=True)

print("X shape:", X.shape)
print("y len:", len(y))
print("Num classes:", len(np.unique(y)))


X shape: (92577, 2049)
y len: 92577
Num classes: 104


## 2) Label encoding

If you already saved a LabelEncoder, load it. Otherwise, rebuild it (fast).


In [3]:
import joblib
from sklearn.preprocessing import LabelEncoder

if LABELS_PATH.exists():
    le = joblib.load(LABELS_PATH)
    print("Loaded LabelEncoder:", LABELS_PATH)
else:
    le = LabelEncoder()
    le.fit(y)
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(le, LABELS_PATH)
    print("Built + saved LabelEncoder:", LABELS_PATH)

y_enc = le.transform(y)
print("Encoded classes:", len(le.classes_))


Loaded LabelEncoder: C:\Projects\Xeno_Canto_Project\artifacts\model_artifacts\label_encoder.joblib
Encoded classes: 104


## 3) Train/Validation/Test split

We use a stratified split for a fair baseline report.


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


Train: (74061, 2049) Val: (9258, 2049) Test: (9258, 2049)


## 4) Load XGBoost model (recommended)

If you already trained an XGBoost model, save it to `ARTIFACTS_DIR/xgb_model.joblib` and it will be loaded here.

If the model file doesn't exist, the notebook includes an **optional training cell** (it can be slow on CPU).


In [8]:
MODEL_PATH = ARTIFACTS_DIR / "lr_multinomial.joblib"

if MODEL_PATH.exists():
    model = joblib.load(MODEL_PATH)
    print("✅ Loaded Logistic Regression model:", MODEL_PATH)
else:
    model = None
    print("❌ Logistic Regression model not found:", MODEL_PATH)



✅ Loaded Logistic Regression model: C:\Projects\Xeno_Canto_Project\artifacts\model_artifacts\lr_multinomial.joblib


## 5) Evaluate (Top‑1 + Top‑5)

For a recommender-style output, **Top‑5 accuracy** is the key metric.


In [9]:
from sklearn.metrics import classification_report, top_k_accuracy_score

assert model is not None, "Model is not loaded. Load it or train it in the optional cell."

proba = model.predict_proba(X_test)
pred = proba.argmax(axis=1)

print(classification_report(y_test, pred, target_names=le.classes_, digits=3))
print("Top-5 accuracy:", float(top_k_accuracy_score(y_test, proba, k=5)))


                               precision    recall  f1-score   support

    acrocephalus arundinaceus      0.598     0.716     0.652        81
       acrocephalus dumetorum      0.448     0.433     0.441        90
       acrocephalus palustris      0.477     0.461     0.469        89
   acrocephalus schoenobaenus      0.552     0.533     0.542        90
      acrocephalus scirpaceus      0.617     0.644     0.630        90
           actitis hypoleucos      0.215     0.289     0.246        90
          aegithalos caudatus      0.346     0.500     0.409        90
              alauda arvensis      0.412     0.544     0.469        90
                  anas crecca      0.240     0.294     0.265        85
           anas platyrhynchos      0.713     0.689     0.701        90
                  anser anser      0.615     0.728     0.667        81
             anthus pratensis      0.277     0.371     0.317        89
             anthus trivialis      0.135     0.056     0.079        90
     

## 6) Demo: Top‑5 prediction for a single `.npz`

Pick any `.npz` file from `artifacts/embeddings_yamnet/<species>/XCxxxxxx.npz`.


In [ ]:


def npz_to_feature(npz_path: Path) -> np.ndarray:
    d = np.load(npz_path, allow_pickle=True)
    emb_mean = d["emb_mean"].astype(np.float32)
    emb_std  = d["emb_std"].astype(np.float32)
    n_frames = np.array([d["embeddings"].shape[0]], dtype=np.float32)
    x = np.concatenate([emb_mean, emb_std, n_frames])
    return x

def predict_top5_from_npz(npz_path: Path):
    x = npz_to_feature(npz_path).reshape(1, -1)
    p = model.predict_proba(x)[0]
    top5_idx = p.argsort()[-5:][::-1]
    return [(le.classes_[i], float(p[i])) for i in top5_idx]

# Example: pick one file manually
EXAMPLE_NPZ = r"C:\Projects\Xeno_Canto_Project\artifacts\embeddings_yamnet\acrocephalus palustris\XC1000241.npz"


if EXAMPLE_NPZ is not None:
    top5 = predict_top5_from_npz(Path(EXAMPLE_NPZ))
    print("Top-5 predictions:")
    for label, score in top5:
        print(f"{label:40s}  {score:.4f}")
else:
    print("Set EXAMPLE_NPZ to a real .npz path to run the demo.")


Top-5 predictions:
sylvia atricapilla                        0.2622
troglodytes troglodytes                   0.0816
hippolais icterina                        0.0702
ficedula hypoleuca                        0.0661
carduelis carduelis                       0.0620


## Insights

- **Transfer learning**: YAMNet turns raw audio into a robust 1024‑D embedding per time frame.
- **Feature engineering**: we summarize each recording using **mean + std** of embeddings (+ number of frames).
- **Model**: XGBoost (multiclass) learns non‑linear boundaries in the embedding space.
- **Business-friendly output**: use **Top‑5** predictions as a recommender to handle acoustic similarity between species.
- **Result**: Top‑5 accuracy around **~0.68** on a held‑out test set (105 species, ~92k recordings).

This is a solid baseline under tight compute/time constraints, and a good foundation for future upgrades (sequence models on frame embeddings, class balancing strategies, and more robust splits).
